> Hermes is built and running in this Deepnote project; the local login page returns HTTP 200 on port \`8080\`\. The public address we tested still returns HTTP 403\. If you enabled Incoming connections, please copy the exact port\-8080 URL shown in this project's Settings → Machine → More options and send it here—the Deepnote\-generated address may differ from the one this runtime exposes\. I can then test that exact URL\. Login username: \`admin\`\. The generated password is stored in \`/datasets/\_deepnote\_work/\.deepnote\-hermes\-data/admin\_password\`; keep that file private\. Keep the project machine running while using the panel\.

In [19]:
from pathlib import Path
import os, subprocess, time

repo_dir = Path.cwd() / "hermes-agent-template"
runtime_dir = Path("/opt/hermes/deepnote_payload")
build_script = repo_dir / "deepnote_build.py"
start_script = repo_dir / "deepnote_start.sh"

if not (runtime_dir / "wrapper-venv/bin/python").is_file() or not (runtime_dir / "bin/hermes").is_file():
    build_log = Path("/tmp/hermes-deepnote-build.log")
    with build_log.open("ab", buffering=0) as log:
        build_process = subprocess.Popen(
            ["python3", str(build_script)], cwd=Path.cwd(),
            env={**os.environ, "PYTHONUNBUFFERED": "1"}, stdin=subprocess.DEVNULL,
            stdout=log, stderr=subprocess.STDOUT, start_new_session=True, close_fds=True,
        )
    print(f"The machine-local Hermes runtime is absent after machine stop; rebuild started (PID {build_process.pid}).")
    print(f"Build log: {build_log}")
else:
    port_probe = subprocess.run(
        ["python3", "-c", "import socket; s=socket.socket(); s.settimeout(.2); r=s.connect_ex(('127.0.0.1',8080)); s.close(); raise SystemExit(0 if r==0 else 1)"],
        capture_output=True,
    )
    if port_probe.returncode != 0:
        server_log = Path("/tmp/hermes-deepnote-server.log")
        with server_log.open("ab", buffering=0) as log:
            server_process = subprocess.Popen(
                ["bash", str(start_script)], cwd=Path.cwd(), env=os.environ.copy(),
                stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT,
                start_new_session=True, close_fds=True,
            )
        print(f"Hermes panel server started (PID {server_process.pid}).")
    else:
        print("Hermes panel server was already listening on port 8080.")

    local_status = "000"
    for _ in range(60):
        time.sleep(1)
        probe = subprocess.run(
            ["curl", "--noproxy", "*", "--connect-timeout", "1", "--max-time", "3", "-sS", "-o", "/dev/null", "-w", "%{http_code}", "http://127.0.0.1:8080/setup/login"],
            text=True, capture_output=True,
        )
        local_status = probe.stdout.strip() or "000"
        if local_status == "200":
            break
    print(f"Local setup/login status: HTTP {local_status}")

    runtime_uuid = os.environ.get("DEEPNOTE_RUNTIME_UUID", "").strip()
    panel_url = f"https://{runtime_uuid}.deepnoteproject.com" if runtime_uuid else ""
    if panel_url and local_status == "200":
        external = subprocess.run(
            ["curl", "--connect-timeout", "5", "--max-time", "12", "-sS", "-o", "/dev/null", "-w", "%{http_code}", panel_url + "/setup/login"],
            text=True, capture_output=True,
        )
        external_status = external.stdout.strip() or "000"
        print(f"External panel login status: HTTP {external_status}")
        if external_status in {"200", "302"}:
            print(f"Verified panel link: {panel_url}/setup/login")
        else:
            print("The service is running, but Deepnote is still blocking incoming requests; check both workspace and project Incoming connections settings.")
    elif not panel_url:
        print("No Deepnote runtime hostname is available in this machine session.")

Hermes panel server was already listening on port 8080.
Local setup/login status: HTTP 200
External panel login status: HTTP 403
The service is running, but Deepnote is still blocking incoming requests; check both workspace and project Incoming connections settings.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=875c7438-df6d-4ade-9a98-991f343c7f98' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>